In [2]:
from urllib.parse import urljoin
import requests

def check_robots_txt(base_url):
    """사이트의 robots.txt 내용을 확인합니다."""
    robots_url = urljoin(base_url, '/robots.txt')
    try:
        resp = requests.get(robots_url, timeout=5)
        if resp.status_code == 200:
            print(f'robots.txt ({robots_url}):')
            print(resp.text[:500])
        else:
            print(f'robots.txt 없음 (상태 코드: {resp.status_code})')
    except Exception as e:
        print(f'확인 실패: {e}')


#check_robots_txt(SITE_URL)
check_robots_txt("https://www.golfzon.com/course/main")

robots.txt (https://www.golfzon.com/robots.txt):
User-agent: * 
Disallow: /myrecord/
Disallow: /tournament/
Disallow: /trophy/
Disallow: /ranking/
Disallow: /holeinone/

Sitemap:https://www.golfzon.com/sitemap/


In [ ]:
# 크롤러를 만들 때 이걸 앞단에 넣어두면 막힌 경로는 자동으로 건너뛰게 할 수 있어요.
# from urllib.robotparser import RobotFileParser

rp = RobotFileParser("https://www.golfzon.com/robots.txt")
rp.read()

print(rp.can_fetch("*", "https://www.golfzon.com/course/main"))   # True
print(rp.can_fetch("*", "https://www.golfzon.com/ranking/"))      # False

True
False


In [5]:
driver = webdriver.Chrome()
driver.get("https://www.golfzon.com/course/main")
time.sleep(5)

with open("debug_list_page.html", "w", encoding="utf-8") as f:
    f.write(driver.page_source)

# '전체코스' 글자가 들어 있는 요소와 그 부모 확인
for el in driver.find_elements(By.XPATH, "//*[contains(text(), '전체코스')]"):
    parent = el.find_element(By.XPATH, "./..")
    print(el.tag_name, el.get_attribute("class"), "| 부모:", parent.tag_name,
          parent.get_attribute("id"), parent.get_attribute("class"))

a router-link-exact-active router-link-active now_path | 부모: li  
a router-link-exact-active router-link-active on | 부모: li  


In [4]:
"""
골프존 전체코스 크롤러 (Selenium)
- 1단계: 전체코스 목록 52페이지를 돌며 골프장 목록 + 상세페이지 링크 수집
- 2단계: 각 골프장 상세페이지에서 홀별 PAR / 거리 수집 → CSV 저장

⚠️ SELECTORS 값은 실제 페이지 구조를 F12로 확인해서 수정해야 합니다.
   먼저 TOTAL_PAGES = 1, DETAIL_LIMIT = 3 으로 테스트한 뒤 전체 실행하세요.
"""
import csv
import os
import re
import time

from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# ───────────────────────── 설정 ─────────────────────────
LIST_URL = "https://www.golfzon.com/course/main"
TOTAL_PAGES = 52          # 테스트할 땐 1
DETAIL_LIMIT = None       # 테스트할 땐 3 (None = 전부)
DELAY = 1.5               # 요청 간격(초) — 너무 줄이지 마세요
HEADLESS = False          # True면 브라우저 창 없이 실행

LINKS_CSV = "golfzon_links.csv"     # 1단계 결과 (중간 저장)
OUTPUT_CSV = "golfzon_courses.csv"  # 최종 결과

SELECTORS = {
    # ── 목록 페이지 ──
    "all_section": "#allCourse",   # '전체코스' 영역 컨테이너 (신규코스 제외용) ← 꼭 확인
    "course_item": "li",           # all_section 안의 골프장 1개
    "region":      ".area",        # 지역
    "name":        ".name",        # 골프장 이름
    "detail_link": "a",            # 상세페이지 링크
    "pagination":  ".paging",      # 페이지 번호 영역
    "next_group":  ".next",        # 페이지 번호 다음 묶음(>) 버튼
    # ── 상세 페이지 ──
    "hole_table":  "table",        # 홀 정보(스코어카드) 표
}

# ───────────────────────── 공통 함수 ─────────────────────────
def make_driver():
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,1000")
    return webdriver.Chrome(options=opts)


def save_html(driver, filename):
    """구조 확인용으로 현재 페이지 HTML 저장"""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(driver.page_source)
    print(f"  [디버그] {filename} 저장")


def safe_text(parent, selector):
    try:
        return parent.find_element(By.CSS_SELECTOR, selector).text.strip()
    except NoSuchElementException:
        return ""


def to_int(s):
    m = re.search(r"\d+", s.replace(",", ""))
    return int(m.group()) if m else None


# ───────────────────────── 1단계: 목록 수집 ─────────────────────────
def get_items(driver):
    section = driver.find_element(By.CSS_SELECTOR, SELECTORS["all_section"])
    return section.find_elements(By.CSS_SELECTOR, SELECTORS["course_item"])


def go_to_page(driver, n):
    """페이지 번호 n 클릭. 안 보이면 '다음 묶음' 버튼을 눌러가며 찾음"""
    for _ in range(10):
        pag = driver.find_element(By.CSS_SELECTOR, SELECTORS["pagination"])
        targets = [a for a in pag.find_elements(By.CSS_SELECTOR, "a")
                   if a.text.strip() == str(n)]
        old_first = get_items(driver)[0]
        if targets:
            driver.execute_script("arguments[0].click();", targets[0])
        else:
            btn = pag.find_element(By.CSS_SELECTOR, SELECTORS["next_group"])
            driver.execute_script("arguments[0].click();", btn)
        try:  # 목록이 새로 그려질 때까지 대기
            WebDriverWait(driver, 10).until(EC.staleness_of(old_first))
        except TimeoutException:
            pass
        time.sleep(DELAY)
        if targets:
            return True
    return False


def collect_links(driver):
    driver.get(LIST_URL)
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, SELECTORS["all_section"])))
    time.sleep(DELAY)
    save_html(driver, "debug_list_page.html")

    rows = []
    for page in range(1, TOTAL_PAGES + 1):
        if page > 1 and not go_to_page(driver, page):
            print(f"  ⚠️ {page}페이지로 이동 실패 — 중단")
            break
        items = get_items(driver)
        for it in items:
            link = ""
            try:
                link = it.find_element(By.CSS_SELECTOR, SELECTORS["detail_link"]).get_attribute("href") or ""
            except NoSuchElementException:
                pass
            rows.append({
                "지역": safe_text(it, SELECTORS["region"]),
                "골프장명": safe_text(it, SELECTORS["name"]),
                "상세URL": link,
                "목록원문": it.text.replace("\n", " / "),  # 셀렉터가 틀려도 정보는 남도록
            })
        print(f"[목록] {page}/{TOTAL_PAGES} 페이지 — {len(items)}개 (누적 {len(rows)})")

    with open(LINKS_CSV, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=["지역", "골프장명", "상세URL", "목록원문"])
        w.writeheader()
        w.writerows(rows)
    return rows


# ───────────────────────── 2단계: 상세 수집 ─────────────────────────
def read_table(table):
    rows = []
    for tr in table.find_elements(By.CSS_SELECTOR, "tr"):
        cells = [c.text.strip() for c in tr.find_elements(By.CSS_SELECTOR, "th, td")]
        if cells:
            rows.append(cells)
    return rows


def parse_holes(rows):
    """스코어카드 표 → [(홀, 파, 거리)]. 가로형/세로형 모두 시도"""
    def find_row(keys):
        for r in rows:
            if any(k in r[0].upper() for k in keys):
                return r
        return None

    hole_r = find_row(["HOLE", "홀"])
    par_r = find_row(["PAR", "파"])
    dist_r = find_row(["거리", "DIST", "METER", "YARD", "(M)", "(Y)"])

    holes = []
    if hole_r and par_r and dist_r:  # 가로형: 행 = HOLE / PAR / 거리
        for i in range(1, min(len(hole_r), len(par_r), len(dist_r))):
            if hole_r[i].isdigit():  # OUT / IN / TOTAL 칸 제외
                holes.append((int(hole_r[i]), to_int(par_r[i]), to_int(dist_r[i])))
        return holes

    header = [h.upper() for h in rows[0]] if rows else []  # 세로형: 열 = HOLE / PAR / 거리
    idx = lambda keys: next((i for i, h in enumerate(header) if any(k in h for k in keys)), None)
    hi, pi, di = idx(["HOLE", "홀"]), idx(["PAR", "파"]), idx(["거리", "DIST", "M", "Y"])
    if None not in (hi, pi, di):
        for r in rows[1:]:
            if len(r) > max(hi, pi, di) and r[hi].isdigit():
                holes.append((int(r[hi]), to_int(r[pi]), to_int(r[di])))
    return holes


def scrape_detail(driver, url, debug=False):
    driver.get(url)
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, SELECTORS["hole_table"])))
    time.sleep(DELAY)
    if debug:
        save_html(driver, "debug_detail_page.html")
    holes = []
    for t in driver.find_elements(By.CSS_SELECTOR, SELECTORS["hole_table"]):
        holes += parse_holes(read_table(t))
    return holes


def main():
    driver = make_driver()
    try:
        # 1단계 (이미 수집했다면 건너뜀)
        if os.path.exists(LINKS_CSV):
            with open(LINKS_CSV, encoding="utf-8-sig") as f:
                links = list(csv.DictReader(f))
            print(f"[목록] {LINKS_CSV} 에서 {len(links)}개 불러옴")
        else:
            links = collect_links(driver)

        # 2단계 (중간에 끊겨도 이어서 진행)
        fields = ["지역", "골프장명", "홀수", "총거리", "PAR3개수", "PAR3거리",
                  "전체홀정보", "상세URL", "목록원문"]
        done = set()
        if os.path.exists(OUTPUT_CSV):
            with open(OUTPUT_CSV, encoding="utf-8-sig") as f:
                done = {r["상세URL"] for r in csv.DictReader(f)}
        new_file = not os.path.exists(OUTPUT_CSV)

        targets = [r for r in links if r["상세URL"] and r["상세URL"] not in done]
        if DETAIL_LIMIT:
            targets = targets[:DETAIL_LIMIT]

        with open(OUTPUT_CSV, "a", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=fields)
            if new_file:
                w.writeheader()
            for i, r in enumerate(targets, 1):
                try:
                    holes = scrape_detail(driver, r["상세URL"], debug=(i == 1))
                except Exception as e:
                    print(f"  ⚠️ 실패: {r['골프장명']} ({type(e).__name__})")
                    continue
                par3 = [(h, d) for h, p, d in holes if p == 3]
                w.writerow({
                    "지역": r["지역"],
                    "골프장명": r["골프장명"],
                    "홀수": len(holes),
                    "총거리": sum(d for _, _, d in holes if d),
                    "PAR3개수": len(par3),
                    "PAR3거리": " / ".join(f"{h}H:{d}" for h, d in par3),
                    "전체홀정보": " | ".join(f"{h}H P{p} {d}" for h, p, d in holes),
                    "상세URL": r["상세URL"],
                    "목록원문": r["목록원문"],
                })
                f.flush()
                print(f"[상세] {i}/{len(targets)} {r['골프장명']} — {len(holes)}홀, PAR3 {len(par3)}개")
    finally:
        driver.quit()
    print(f"\n완료 → {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

TimeoutException: Message: 
Stacktrace:
0   chromedriver                        0x00000001020486fa chromedriver + 7001850
1   chromedriver                        0x0000000101a10365 chromedriver + 480101
2   chromedriver                        0x0000000101a6536b chromedriver + 828267
3   chromedriver                        0x0000000101a656b1 chromedriver + 829105
4   chromedriver                        0x0000000101ab1a74 chromedriver + 1141364
5   chromedriver                        0x0000000101aaebdf chromedriver + 1129439
6   chromedriver                        0x0000000101a58944 chromedriver + 776516
7   chromedriver                        0x0000000101a59711 chromedriver + 780049
8   chromedriver                        0x0000000102008097 chromedriver + 6738071
9   chromedriver                        0x000000010200c596 chromedriver + 6755734
10  chromedriver                        0x0000000101fe8b14 chromedriver + 6609684
11  chromedriver                        0x000000010200d012 chro

In [6]:
import json
from selenium.webdriver.chrome.options import Options

opts = Options()
opts.set_capability("goog:loggingPrefs", {"performance": "ALL"})
driver = webdriver.Chrome(options=opts)
driver.get("https://www.golfzon.com/course/main")
time.sleep(5)

for entry in driver.get_log("performance"):
    msg = json.loads(entry["message"])["message"]
    if msg["method"] == "Network.responseReceived":
        res = msg["params"]["response"]
        if "json" in res.get("mimeType", ""):
            print(res["url"])

https://adservice.golfzon.com/adp/banners/request?adSlotCode=E8626852D9D6
https://adservice.golfzon.com/adp/banners/request?adSlotCode=807418B78CD8
https://adservice.golfzon.com/adp/banners/request?adSlotCode=1B2C3F0E29DE
https://lobby.golfzon.com/v1/dotcom/courses/course:new
https://lobby.golfzon.com/v1/dotcom/courses/course/search/list?page=1&softwareType=0&orderType=1&searchWord=&ciVersion=0&areaNo=0
https://adservice.golfzon.com/adp/banners/impressions?adTargetCode=19722AE96DE6402
https://adservice.golfzon.com/adp/banners/impressions?adTargetCode=848F1F2324A6401
https://lobby.golfzon.com/v1/dotcom/courses/course/top10


In [7]:
import requests, json

url = "https://lobby.golfzon.com/v1/dotcom/courses/course/search/list"
params = {"page": 1, "softwareType": 0, "orderType": 1,
          "searchWord": "", "ciVersion": 0, "areaNo": 0}
headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.golfzon.com/",
    "Origin": "https://www.golfzon.com",
}

res = requests.get(url, params=params, headers=headers, timeout=10)
print(res.status_code)
data = res.json()
print(json.dumps(data, ensure_ascii=False, indent=2)[:2000])

200
[
  {
    "ciCode": 100000389,
    "ccName": "만쥬 GC",
    "difficultyGreen": 8,
    "difficultyCc": 8,
    "address": "일본",
    "logoImage": "http://uf.gzcdn.net/tournament/clientimages/202608/2414/41/8bee66cea96e4125ba6fe64e41babd53.jpg",
    "boardCount": 0,
    "ciVersion": 2,
    "district": 0,
    "country": 2,
    "holeCount": 18,
    "parCount": 72,
    "totalWidth": 6348,
    "isVisionPlus": "Y",
    "isNgs": "Y",
    "isTvnx": "Y",
    "isNxp": "Y",
    "ciNight": 0,
    "realityYn": "N",
    "difficultyYn": "N",
    "isPopular": null,
    "isFavorite": null,
    "membership": "Y"
  },
  {
    "ciCode": 100001088,
    "ccName": "신요 CC",
    "difficultyGreen": 8,
    "difficultyCc": 8,
    "address": "일본",
    "logoImage": "http://uf.gzcdn.net/tournament/clientimages/202608/2414/39/ea06089a24f14e6c86f1f5d4b991932c.jpg",
    "boardCount": 0,
    "ciVersion": 2,
    "district": 0,
    "country": 2,
    "holeCount": 18,
    "parCount": 72,
    "totalWidth": 6370,
    "isVision

In [9]:
for entry in driver.get_log("performance"):
    msg = json.loads(entry["message"])["message"]
    if msg["method"] == "Network.responseReceived":
        u = msg["params"]["response"]["url"]
        if "lobby.golfzon.com" in u:
            print(u)

In [10]:
import time
import pandas as pd

all_rows = []
for page in range(1, 100):            # 안전장치: 최대 99페이지
    params["page"] = page
    items = requests.get(url, params=params, headers=headers, timeout=10).json()
    if not items:                     # 빈 리스트 = 마지막 페이지 지남
        break
    all_rows.extend(items)
    print(f"{page}페이지: {len(items)}개 (누적 {len(all_rows)})")
    time.sleep(1)

df = pd.DataFrame(all_rows)
df.to_csv("golfzon_list_raw.csv", index=False, encoding="utf-8-sig")   # 원본 전체 컬럼

cols = {"ciCode": "코스ID", "ccName": "골프장명", "address": "지역",
        "holeCount": "홀수", "parCount": "총파", "totalWidth": "총거리"}
df[list(cols)].rename(columns=cols).to_csv("golfzon_list.csv", index=False, encoding="utf-8-sig")
print("완료:", len(df), "개")

1페이지: 10개 (누적 10)
2페이지: 10개 (누적 20)
3페이지: 10개 (누적 30)
4페이지: 10개 (누적 40)
5페이지: 10개 (누적 50)
6페이지: 10개 (누적 60)
7페이지: 10개 (누적 70)
8페이지: 10개 (누적 80)
9페이지: 10개 (누적 90)
10페이지: 10개 (누적 100)
11페이지: 10개 (누적 110)
12페이지: 10개 (누적 120)
13페이지: 10개 (누적 130)
14페이지: 10개 (누적 140)
15페이지: 10개 (누적 150)
16페이지: 10개 (누적 160)
17페이지: 10개 (누적 170)
18페이지: 10개 (누적 180)
19페이지: 10개 (누적 190)
20페이지: 10개 (누적 200)
21페이지: 10개 (누적 210)
22페이지: 10개 (누적 220)
23페이지: 10개 (누적 230)
24페이지: 10개 (누적 240)
25페이지: 10개 (누적 250)
26페이지: 10개 (누적 260)
27페이지: 10개 (누적 270)
28페이지: 10개 (누적 280)
29페이지: 10개 (누적 290)
30페이지: 10개 (누적 300)
31페이지: 10개 (누적 310)
32페이지: 10개 (누적 320)
33페이지: 10개 (누적 330)
34페이지: 10개 (누적 340)
35페이지: 10개 (누적 350)
36페이지: 10개 (누적 360)
37페이지: 10개 (누적 370)
38페이지: 10개 (누적 380)
39페이지: 10개 (누적 390)
40페이지: 10개 (누적 400)
41페이지: 10개 (누적 410)
42페이지: 10개 (누적 420)
43페이지: 10개 (누적 430)
44페이지: 10개 (누적 440)
45페이지: 10개 (누적 450)
46페이지: 10개 (누적 460)
47페이지: 10개 (누적 470)
48페이지: 10개 (누적 480)
49페이지: 10개 (누적 490)
50페이지: 10개 (누적 500)
51페이지: 10개 (누적 510

In [11]:
import json
from selenium.webdriver.chrome.options import Options

opts = Options()
opts.set_capability("goog:loggingPrefs", {"performance": "ALL"})
driver = webdriver.Chrome(options=opts)
driver.get("https://www.golfzon.com/course/detail/102490370")
time.sleep(5)

for entry in driver.get_log("performance"):
    msg = json.loads(entry["message"])["message"]
    if msg["method"] == "Network.responseReceived":
        res = msg["params"]["response"]
        if "lobby.golfzon.com" in res["url"]:
            print("▶", res["url"])
            try:
                body = driver.execute_cdp_cmd("Network.getResponseBody",
                                              {"requestId": msg["params"]["requestId"]})
                print(body["body"][:300], "\n")
            except Exception:
                print("  (응답 본문 없음)\n")

▶ https://lobby.golfzon.com/v1/dotcom/courses/course/102490370/details
  (응답 본문 없음)

▶ https://lobby.golfzon.com/v1/dotcom/courses/course/102490370/details
{"ciCode":102490370,"ccName":"골프존카운티 선산","createdAt":"Y","country":1,"district":5,"address":"경상북도 구미시 산동면 강동로 953-73","tel":"054-473-6200","fax":"054-473-6213","homepageUrl":"http://www.golfzoncounty.com/","description":"<p><span style=\"color: rgb(51, 153, 0);\"><b style=\"\"><span style=\"font-siz 

▶ https://lobby.golfzon.com/v1/dotcom/courses/course/top10
  (응답 본문 없음)

▶ https://lobby.golfzon.com/v1/dotcom/courses/course/top10
[{"ccName":"스카이뷰","ciCode":100000168,"rank":1},{"ccName":"블루원 상주 골프리조트","ciCode":100001028,"rank":2},{"ccName":"플라밍고 CC - 듄스/리버","ciCode":100001063,"rank":3},{"ccName":"용인 CC","ciCode":100000229,"rank":4},{"ccName":"오션힐스 영천 CC - 달/해","ciCode":100001066,"rank":5},{"ccName":"오션힐스 영천 CC - 해/별","ciCode" 

▶ https://lobby.golfzon.com/v1/dotcom/courses/course/102490370/details/best-top5
  (응답 본문 없음)

▶ https://l

In [14]:
"""
골프존 골프장 상세정보 크롤러 (requests 전용, Selenium 불필요)
- 입력: golfzon_list_raw.csv (목록 API로 만든 514개 골프장)
- 골프장마다 API 2개 호출
    /details            → 기본정보 (주소, 전화, 홈페이지 등)
    /details/hole-info  → 홀별 파, 티별 거리, 공략 설명
- 출력
    golfzon_detail.csv  → 골프장 1개 = 1행 (PAR3 개수·거리, 티별 합계, 기본정보)
    golfzon_holes.csv   → 홀 1개 = 1행 (홀별 상세, 공략 설명 포함)

먼저 LIMIT = 5 로 테스트 → 결과 확인 후 None 으로 바꿔 전체 실행
"""
import json
import os
import time

import pandas as pd
import requests

# ───────────────────────── 설정 ─────────────────────────
INPUT_CSV = "golfzon_list_raw.csv"
COURSE_CSV = "golfzon_detail.csv"
HOLE_CSV = "golfzon_holes.csv"
CACHE_DIR = "cache_detail"     # 골프장별 원본 JSON 저장 (재실행 시 재요청 안 함)
LIMIT = None                      # 테스트용. 전체는 None
DELAY = 0.5                    # 요청 간격(초)
DIST_KEY = "backTee"           # PAR3 거리에 쓸 티 (champTee / backTee / frontTee ...)
DROP_KEYS = {"description"}    # 기본정보에서 뺄 키 (HTML이라 길고 지저분함)

BASE = "https://lobby.golfzon.com/v1/dotcom/courses/course/{ci}"
APIS = {
    "details": BASE + "/details",
    "holes": BASE + "/details/hole-info",
}
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.golfzon.com/",
    "Origin": "https://www.golfzon.com",
}


# ───────────────────────── 수집 ─────────────────────────
def get_json(ci):
    """골프장 1개의 API 응답들. 캐시가 있으면 파일에서 읽음"""
    path = os.path.join(CACHE_DIR, f"{ci}.json")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f), False
    data = {}
    for name, url in APIS.items():
        r = requests.get(url.format(ci=ci), headers=HEADERS, timeout=10)
        r.raise_for_status()
        data[name] = r.json()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    return data, True


# ───────────────────────── 가공 ─────────────────────────
def parse_holes(ci, holes_json):
    """holeInfoList(서브코스별 리스트의 리스트) → 홀 1개 = dict 1개"""
    parts = (holes_json or {}).get("holeInfoList") or []
    rows = []
    for part_no, part in enumerate(parts, 1):
        for i, h in enumerate(part, 1):
            label = f"{part_no}-{i}" if len(parts) > 1 else str(i)
            rows.append({"ciCode": ci, "서브코스": part_no, "홀": label, **h})
    return rows


def summarize(holes):
    par3 = [h for h in holes if h.get("basicPar") == 3]
    tee_keys = sorted({k for h in holes for k in h if k.endswith("Tee")})
    out = {
        "인식홀수": len(holes),
        "PAR3개수": len(par3),
        "PAR4개수": sum(1 for h in holes if h.get("basicPar") == 4),
        "PAR5개수": sum(1 for h in holes if h.get("basicPar") == 5),
        f"PAR3거리({DIST_KEY})": " / ".join(f"{h['홀']}H:{h.get(DIST_KEY)}" for h in par3),
        "전체홀정보": " | ".join(f"{h['홀']}H P{h.get('basicPar')} {h.get(DIST_KEY)}" for h in holes),
    }
    for t in tee_keys:  # 티별 거리 합계 → 목록의 총거리와 비교해 단위·기준 확인용
        out[f"합계_{t}"] = sum(h[t] for h in holes if isinstance(h.get(t), (int, float)))
    return out


def details_info(details_json):
    if not isinstance(details_json, dict):
        return {}
    return {f"상세_{k}": v for k, v in details_json.items()
            if k not in DROP_KEYS and not isinstance(v, (list, dict))}


# ───────────────────────── 실행 ─────────────────────────
def main():
    os.makedirs(CACHE_DIR, exist_ok=True)
    base = pd.read_csv(INPUT_CSV)
    if LIMIT:
        base = base.head(LIMIT)

    course_rows, hole_rows = [], []
    for i, (ci, name) in enumerate(zip(base["ciCode"], base["ccName"]), 1):
        try:
            data, requested = get_json(ci)
        except Exception as e:
            print(f"  ⚠️ [{i}] {name} 실패: {type(e).__name__} {e}")
            continue
        holes = parse_holes(ci, data.get("holes"))
        hole_rows += holes
        course_rows.append({"ciCode": ci, **summarize(holes), **details_info(data.get("details"))})
        print(f"[{i}/{len(base)}] {name} — {len(holes)}홀, PAR3 {course_rows[-1]['PAR3개수']}개")
        if requested:
            time.sleep(DELAY)

    # 골프장 단위 CSV: 목록 정보 + 요약 + 기본정보
    list_cols = {"ciCode": "코스ID", "ccName": "골프장명", "address": "지역",
                 "holeCount": "홀수", "parCount": "총파", "totalWidth": "총거리"}
    course = base[list(list_cols)].merge(pd.DataFrame(course_rows), on="ciCode", how="left")
    course.rename(columns=list_cols).to_csv(COURSE_CSV, index=False, encoding="utf-8-sig")

    # 홀 단위 CSV
    holes = pd.DataFrame(hole_rows)
    if not holes.empty:
        holes = base[["ciCode", "ccName"]].merge(holes, on="ciCode")
        holes.rename(columns={"ciCode": "코스ID", "ccName": "골프장명"}).to_csv(
            HOLE_CSV, index=False, encoding="utf-8-sig")

    print(f"\n완료 → {COURSE_CSV} ({len(course)}개 골프장), {HOLE_CSV} ({len(holes)}개 홀)")


if __name__ == "__main__":
    main()

[1/514] 만쥬 GC — 18홀, PAR3 4개
[2/514] 신요 CC — 18홀, PAR3 4개
[3/514] 이와테 누마쿠나이 CC — 18홀, PAR3 4개
[4/514] 카루이자와 코겐 GC — 18홀, PAR3 5개
[5/514] 나라시노 CC - KING — 18홀, PAR3 4개


[6/514] 나라시노 CC - QUEEN — 18홀, PAR3 4개
[7/514] 소부 CC - 소부 코스 — 18홀, PAR3 4개
[8/514] 요넥스 CC — 18홀, PAR3 4개
[9/514] 칠곡 아이위시 CC — 18홀, PAR3 5개
[10/514] 군산 CC - 토너먼트 — 18홀, PAR3 4개
[11/514] 군산 CC - 전주/익산 — 18홀, PAR3 4개
[12/514] 바나힐스 골프클럽 — 18홀, PAR3 4개
[13/514] 블루캐니언 CC - 레이크 — 18홀, PAR3 4개
[14/514] 블루캐니언 CC - 캐니언 — 18홀, PAR3 4개
[15/514] 수서 골프&리조트 — 18홀, PAR3 3개
[16/514] 일라이트 CC — 18홀, PAR3 4개
[17/514] 오션힐스 영천 CC - 달/해 — 18홀, PAR3 4개
[18/514] 오션힐스 영천 CC - 별/달 — 18홀, PAR3 4개
[19/514] 오션힐스 영천 CC - 해/별 — 18홀, PAR3 4개
[20/514] 중원 GC - 고구려/백제 — 18홀, PAR3 5개
[21/514] 중원 GC - 백제/신라 — 18홀, PAR3 4개
[22/514] 중원 GC - 신라/고구려 — 18홀, PAR3 5개
[23/514] AI 천문봉 CC — 18홀, PAR3 4개
[24/514] 그레이트 오크스 CC — 18홀, PAR3 4개
[25/514] 다이 하코네 CC — 18홀, PAR3 4개
[26/514] 더 클래식 GC — 18홀, PAR3 4개
[27/514] 레이크 머세드 GC — 18홀, PAR3 4개
[28/514] 레이크 포레스트 - 센추리 — 18홀, PAR3 4개
[29/514] 세고비아 GC 인 치요다 — 18홀, PAR3 4개
[30/514] 세인트 앤드류스 - 뉴 코스 — 18홀, PAR3 4개
[31/514] 세인트 앤드류스 - 쥬빌리 코스 — 18홀, PAR3 4개
[32/514] 오키드 CC — 27홀, PAR3 6개
[33/51